# f033_f019_mlp_s7: Submission notebook

Frozen 2019-2023 fit with fixed 2024 validation. Upload `f033_weights.json` beside this notebook.

In [ ]:
def main(datasources, start_date, end_date):
    """Frozen low-correlation summary MLP inference for the AI factor track.

    The model was fitted on 2019-2023 data and selected on a fixed 2024
    validation set. This function only builds contemporaneous primitives and
    runs the frozen 20-session model over the interval supplied by the platform.
    """
    import gc
    import json
    import os

    import dai
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn.functional as torch_functional

    MIN_BARS = 60
    T = 20
    PRIM_COLS = [
        "pv_ret_oc",
        "pv_ret_am",
        "pv_ret_pm",
        "pv_ret_first30",
        "pv_ret_last30",
        "pv_pos_close",
        "pv_vwap_dev",
        "pv_vwap_dev_last30",
        "pv_range",
        "rd_std",
        "rd_skew",
        "rd_kurt",
        "rd_upshare",
        "rd_trend_eff",
        "vt_hhi",
        "vt_first30_share",
        "vt_last30_share",
        "vt_ret_vol_corr",
        "vt_absret_vol_corr",
        "vt_avg_trade_size",
        "vt_amihud",
        "vt_vw_ret",
        "ob_ofi1",
        "ob_ofi5",
        "ob_dimb_mean",
        "ob_dimb_std",
        "ob_dimb_last30",
        "ob_dimb1_mean",
        "ob_oimb_mean",
        "ob_ordsize_imb",
        "ob_spread_mean",
        "ob_spread_std",
        "ob_spread_last30",
        "ob_mpd_mean",
        "ob_slope_asym",
        "ob_quote_int",
        "ob_book_to_flow"
    ]

    def month_chunks(d0, d1):
        """[d0,d1] 切自然月分块,返回 [(start,end)] 字符串日期对。"""
        s, e = pd.Timestamp(d0), pd.Timestamp(d1)
        out, cur = [], s
        while cur <= e:
            me = min(cur + pd.offsets.MonthEnd(0), e)
            if me < cur:
                me = min(cur + pd.offsets.MonthEnd(1), e)
            out.append((cur.strftime("%Y-%m-%d"), me.strftime("%Y-%m-%d")))
            cur = me + pd.Timedelta(days=1)
        return out


    def primitive_sql(bar_table, inst_table, d0, d1):
        """单个月度分块的原语充分统计量 SQL:分钟表 → (instrument, d) 一行聚合。"""
        return f"""
        WITH pool AS (
            SELECT DISTINCT date::DATE AS pd, instrument
            FROM {inst_table}
            WHERE date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'
        ),
        base AS (
            SELECT t.instrument, t.date, t.date::DATE AS d,
                   t.open, t.high, t.low, t.close,
                   t.volume::DOUBLE AS volume, t.amount::DOUBLE AS amount,
                   t.deal_number::DOUBLE AS deal_number,
                   t.ask_price1 AS ap1, t.bid_price1 AS bp1,
                   t.ask_price5 AS ap5, t.bid_price5 AS bp5,
                   t.ask_volume1::DOUBLE AS av1, t.bid_volume1::DOUBLE AS bv1,
                   -- Cast each order-book column before summation.
                   (t.bid_volume1::DOUBLE + t.bid_volume2::DOUBLE + t.bid_volume3::DOUBLE
                    + t.bid_volume4::DOUBLE + t.bid_volume5::DOUBLE) AS bidv5,
                   (t.ask_volume1::DOUBLE + t.ask_volume2::DOUBLE + t.ask_volume3::DOUBLE
                    + t.ask_volume4::DOUBLE + t.ask_volume5::DOUBLE) AS askv5,
                   (t.bid_num_orders1::DOUBLE + t.bid_num_orders2::DOUBLE + t.bid_num_orders3::DOUBLE
                    + t.bid_num_orders4::DOUBLE + t.bid_num_orders5::DOUBLE) AS bidn5,
                   (t.ask_num_orders1::DOUBLE + t.ask_num_orders2::DOUBLE + t.ask_num_orders3::DOUBLE
                    + t.ask_num_orders4::DOUBLE + t.ask_num_orders5::DOUBLE) AS askn5
            FROM {bar_table} t
            INNER JOIN pool p ON p.instrument = t.instrument AND p.pd = t.date::DATE
            WHERE t.date::DATE BETWEEN DATE '{d0}' AND DATE '{d1}'
        ),
        w AS (
            SELECT *,
                   LAG(close)       OVER win AS close_p,
                   LAG(volume)      OVER win AS volume_p,
                   LAG(amount)      OVER win AS amount_p,
                   LAG(deal_number) OVER win AS deal_p,
                   LAG(bp1)         OVER win AS bp1_p,
                   LAG(ap1)         OVER win AS ap1_p,
                   LAG(bv1)         OVER win AS bv1_p,
                   LAG(av1)         OVER win AS av1_p,
                   LAG(bidv5)       OVER win AS bidv5_p,
                   LAG(askv5)       OVER win AS askv5_p,
                   ROW_NUMBER()     OVER win AS rn,
                   COUNT(*)         OVER (PARTITION BY instrument, d) AS nb
            FROM base
            WINDOW win AS (PARTITION BY instrument, d ORDER BY date, open, high, low, close, volume, amount, deal_number, ap1, bp1, ap5, bp5, av1, bv1, bidv5, askv5, bidn5, askn5)
        ),
        m AS (
            SELECT instrument, d, rn, nb, (nb - rn + 1) AS rn_desc,
                   CASE WHEN hour(date) < 12 THEN 1 ELSE 0 END AS is_am,
                   open, high, low, close,
                   COALESCE(ln(NULLIF(close, 0) / NULLIF(close_p, 0)), 0)   AS r,
                   GREATEST(volume - COALESCE(volume_p, 0), 0)              AS v,
                   GREATEST(amount - COALESCE(amount_p, 0), 0)              AS a,
                   GREATEST(deal_number - COALESCE(deal_p, 0), 0)           AS tn,
                   CASE WHEN bp1 > 0 AND ap1 > bp1 THEN 1 ELSE 0 END        AS bok,
                   (ap1 + bp1) / 2.0                                        AS mid,
                   ap1, bp1, ap5, bp5, av1, bv1, bidv5, askv5, bidn5, askn5,
                   bp1_p, ap1_p, bv1_p, av1_p, bidv5_p, askv5_p
            FROM w
        ),
        e AS (  -- 每分钟派生量;无效盘口(bok=0)置 NULL,聚合自动忽略
            SELECT instrument, d, rn, rn_desc, nb, is_am, open, high, low, close, r, v, a, tn,
                CASE WHEN bok = 1 THEN (bidv5 - askv5) / NULLIF(bidv5 + askv5, 0) END AS dimb,
                CASE WHEN bok = 1 THEN (bv1 - av1) / NULLIF(bv1 + av1, 0) END         AS dimb1,
                CASE WHEN bok = 1 THEN (bidn5 - askn5) / NULLIF(bidn5 + askn5, 0) END AS oimb,
                CASE WHEN bok = 1 AND bidv5 > 0 AND askv5 > 0 AND bidn5 > 0 AND askn5 > 0
                     THEN ln((bidv5 / bidn5) / NULLIF(askv5 / askn5, 0)) END          AS ordsz,
                CASE WHEN bok = 1 THEN (ap1 - bp1) / NULLIF(mid, 0) END               AS spr,
                CASE WHEN bok = 1 AND bv1 + av1 > 0
                     THEN (ap1*bv1 + bp1*av1) / NULLIF(bv1 + av1, 0) / NULLIF(mid, 0) - 1 END AS mpd,
                CASE WHEN bok = 1 AND bp5 > 0 AND ap5 > 0
                     THEN (ap5 - ap1) / NULLIF(mid, 0) - (bp1 - bp5) / NULLIF(mid, 0) END     AS slope_asym,
                CASE WHEN bok = 1 THEN bv1 + av1 END                                  AS depth1,
                CASE WHEN bok = 1 THEN bidv5 + askv5 END                              AS depth5,
                CASE WHEN bok = 1 AND bp1_p > 0 AND ap1_p > bp1_p THEN
                      (CASE WHEN bp1 >= bp1_p THEN bv1   ELSE 0 END)
                    - (CASE WHEN bp1 <= bp1_p THEN bv1_p ELSE 0 END)
                    - (CASE WHEN ap1 <= ap1_p THEN av1   ELSE 0 END)
                    + (CASE WHEN ap1 >= ap1_p THEN av1_p ELSE 0 END) END              AS ofi1,
                CASE WHEN bok = 1 AND bidv5_p IS NOT NULL AND bp1_p > 0
                     THEN (bidv5 - bidv5_p) - (askv5 - askv5_p) END                   AS ofi5,
                CASE WHEN bp1_p IS NOT NULL AND (bp1 <> bp1_p OR ap1 <> ap1_p)
                     THEN 1 ELSE 0 END                                                AS qchg
            FROM m
        )
        SELECT instrument, d,
            MAX(nb) AS nb,
            SUM(CASE WHEN dimb IS NOT NULL THEN 1 ELSE 0 END)          AS n_book,
            MAX(CASE WHEN rn = 1 THEN open END)                        AS open_first,
            MAX(CASE WHEN rn = nb THEN close END)                      AS close_last,
            MAX(high) AS hi, MIN(low) AS lo,
            SUM(r) AS s_r, SUM(r*r) AS s_r2, SUM(r*r*r) AS s_r3, SUM(r*r*r*r) AS s_r4,
            SUM(ABS(r)) AS s_absr,
            SUM(CASE WHEN r > 0 THEN 1 ELSE 0 END)                     AS n_up,
            SUM(CASE WHEN r <> 0 THEN 1 ELSE 0 END)                    AS n_nz,
            SUM(v) AS s_v, SUM(v*v) AS s_v2, SUM(a) AS s_a, SUM(tn) AS s_tn,
            SUM(r*v) AS s_rv, SUM(ABS(r)*v) AS s_absrv,
            SUM(close*v) AS s_cv,
            SUM(CASE WHEN rn_desc <= 30 THEN close*v ELSE 0 END)       AS cv_last30,
            SUM(CASE WHEN rn <= 30 THEN v ELSE 0 END)                  AS v_first30,
            SUM(CASE WHEN rn_desc <= 30 THEN v ELSE 0 END)             AS v_last30,
            SUM(CASE WHEN rn <= 30 THEN r ELSE 0 END)                  AS r_first30,
            SUM(CASE WHEN rn_desc <= 30 THEN r ELSE 0 END)             AS r_last30,
            SUM(CASE WHEN is_am = 1 THEN r ELSE 0 END)                 AS r_am,
            SUM(CASE WHEN is_am = 0 THEN r ELSE 0 END)                 AS r_pm,
            SUM(ofi1) AS ofi1_sum, SUM(ofi5) AS ofi5_sum,
            AVG(depth1) AS depth1_avg, AVG(depth5) AS depth5_avg,
            SUM(dimb) AS s_dimb, SUM(dimb*dimb) AS s_dimb2,
            AVG(CASE WHEN rn_desc <= 30 THEN dimb END)                 AS dimb_last30,
            AVG(dimb1) AS dimb1_avg, AVG(oimb) AS oimb_avg, AVG(ordsz) AS ordsz_avg,
            SUM(spr) AS s_spr, SUM(spr*spr) AS s_spr2,
            SUM(CASE WHEN spr IS NOT NULL THEN 1 ELSE 0 END)           AS n_spr,
            AVG(CASE WHEN rn_desc <= 30 THEN spr END)                  AS spr_last30,
            AVG(mpd) AS mpd_avg, AVG(slope_asym) AS slope_asym_avg,
            SUM(qchg) AS qchg_sum
        FROM e
        GROUP BY instrument, d
        """


    def finish_primitives(g):
        """充分统计量 → 32 个日频原语(全 pandas 向量化,含全部除零/退化保护)。"""
        eps = 1e-12
        n = g["nb"].astype("float64")
        nbk = g["n_book"].astype("float64")

        def safe_div(a, b):
            b = np.asarray(b, dtype="float64")
            return np.where(np.abs(b) > eps, np.asarray(a, dtype="float64") / np.where(np.abs(b) > eps, b, 1.0), np.nan)

        out = pd.DataFrame({
            "date": pd.to_datetime(g["d"]),
            "instrument": g["instrument"].astype(str),
        })

        # ---- 分钟收益矩 ----
        mu = g["s_r"] / n
        m2 = (g["s_r2"] / n - mu ** 2).clip(lower=0.0)
        sd = np.sqrt(m2)
        m3 = g["s_r3"] / n - 3 * mu * g["s_r2"] / n + 2 * mu ** 3
        m4 = g["s_r4"] / n - 4 * mu * g["s_r3"] / n + 6 * mu ** 2 * g["s_r2"] / n - 3 * mu ** 4

        ok_px = (g["open_first"] > 0) & (g["close_last"] > 0)
        ret_oc = pd.Series(np.where(ok_px, np.log(g["close_last"].where(ok_px, 1.0) / g["open_first"].where(ok_px, 1.0)), np.nan), index=g.index)

        # ---- 价格路径 ----
        out["pv_ret_oc"] = ret_oc
        out["pv_ret_am"] = g["r_am"]
        out["pv_ret_pm"] = g["r_pm"]
        out["pv_ret_first30"] = g["r_first30"]
        out["pv_ret_last30"] = g["r_last30"]
        rng = g["hi"] - g["lo"]
        out["pv_pos_close"] = np.where(rng > eps, (g["close_last"] - g["lo"]) / np.where(rng > eps, rng, 1.0), 0.5)
        vwap = safe_div(g["s_cv"], g["s_v"])
        out["pv_vwap_dev"] = safe_div(g["close_last"], vwap) - 1.0
        vwap_l30 = safe_div(g["cv_last30"], g["v_last30"])
        out["pv_vwap_dev_last30"] = safe_div(vwap_l30, vwap) - 1.0
        out["pv_range"] = safe_div(rng, vwap)

        # ---- 分钟收益分布 ----
        out["rd_std"] = sd
        out["rd_skew"] = np.where(sd > eps, m3 / np.where(sd > eps, sd ** 3, 1.0), np.nan)
        out["rd_kurt"] = np.where(m2 > eps, m4 / np.where(m2 > eps, m2 ** 2, 1.0) - 3.0, np.nan)
        out["rd_upshare"] = np.where(g["n_nz"] > 0, g["n_up"] / np.where(g["n_nz"] > 0, g["n_nz"], 1.0), 0.5)
        out["rd_trend_eff"] = np.abs(ret_oc) / (g["s_absr"] + eps)

        # ---- 量能时序 ----
        out["vt_hhi"] = n * safe_div(g["s_v2"], g["s_v"] ** 2)
        out["vt_first30_share"] = safe_div(g["v_first30"], g["s_v"])
        out["vt_last30_share"] = safe_div(g["v_last30"], g["s_v"])
        mv = g["s_v"] / n
        sdv = np.sqrt((g["s_v2"] / n - mv ** 2).clip(lower=0.0))
        cov_rv = g["s_rv"] / n - mu * mv
        out["vt_ret_vol_corr"] = np.where((sd > eps) & (sdv > eps), cov_rv / (sd * sdv + eps), np.nan)
        mabs = g["s_absr"] / n
        sd_abs = np.sqrt((g["s_r2"] / n - mabs ** 2).clip(lower=0.0))
        cov_av = g["s_absrv"] / n - mabs * mv
        out["vt_absret_vol_corr"] = np.where((sd_abs > eps) & (sdv > eps), cov_av / (sd_abs * sdv + eps), np.nan)
        out["vt_avg_trade_size"] = np.where((g["s_v"] > 0) & (g["s_tn"] > 0),
                                            np.log(np.where(g["s_tn"] > 0, g["s_v"] / np.where(g["s_tn"] > 0, g["s_tn"], 1.0), 1.0)), np.nan)
        out["vt_amihud"] = np.abs(ret_oc) / (g["s_a"] / 1e8 + eps)
        out["vt_vw_ret"] = safe_div(g["s_rv"], g["s_v"])

        # ---- 五档盘口 ----
        out["ob_ofi1"] = safe_div(g["ofi1_sum"], nbk * g["depth1_avg"])
        out["ob_ofi5"] = safe_div(g["ofi5_sum"], nbk * g["depth5_avg"])
        dimb_avg = safe_div(g["s_dimb"], nbk)
        out["ob_dimb_mean"] = dimb_avg
        out["ob_dimb_std"] = np.sqrt(np.clip(safe_div(g["s_dimb2"], nbk) - dimb_avg ** 2, 0.0, None))
        out["ob_dimb_last30"] = g["dimb_last30"]
        out["ob_dimb1_mean"] = g["dimb1_avg"]
        out["ob_oimb_mean"] = g["oimb_avg"]
        out["ob_ordsize_imb"] = g["ordsz_avg"]
        spr_avg = safe_div(g["s_spr"], g["n_spr"])
        out["ob_spread_mean"] = spr_avg
        out["ob_spread_std"] = np.sqrt(np.clip(safe_div(g["s_spr2"], g["n_spr"]) - spr_avg ** 2, 0.0, None))
        out["ob_spread_last30"] = g["spr_last30"]
        out["ob_mpd_mean"] = g["mpd_avg"]
        out["ob_slope_asym"] = g["slope_asym_avg"]
        out["ob_quote_int"] = safe_div(g["qchg_sum"], n)
        out["ob_book_to_flow"] = np.where((g["depth5_avg"] > 0),
                                          np.log(np.where(g["depth5_avg"] > 0, g["depth5_avg"], 1.0) / (g["s_v"] / n + 1.0)), np.nan)

        for c in PRIM_COLS:
            out[c] = pd.to_numeric(out[c], errors="coerce").astype("float32")
            out[c] = out[c].replace([np.inf, -np.inf], np.nan)
        return out[["date", "instrument"] + PRIM_COLS]


    def build_primitives(q, bar_table, inst_table, d0, d1, min_bars=MIN_BARS):
        """按月分块拉充分统计量 → finish → 拼接全期原语宽表。q(sql, filters) -> DataFrame。"""
        try:
            import psutil

            def _avail():
                return f"{psutil.virtual_memory().available / 2**30:.1f}GB"
        except Exception:
            def _avail():
                return "n/a"

        chunks = month_chunks(d0, d1)
        parts = []
        for i, (cs, ce) in enumerate(chunks):
            stats = q(primitive_sql(bar_table, inst_table, cs, ce),
                      {"date": [cs, ce + " 23:59:59"]})
            if len(stats) == 0:
                print(f"[f006-step1] 分块 {i+1}/{len(chunks)} ({cs}..{ce}): 0 行,跳过")
                continue
            # DAI 的 SUM(整数列) 返回 HUGEINT/DECIMAL → .df() 是 object 列(Decimal),
            # 直接参与算术会 TypeError(an earlier platform run 平台实测;本地 DuckDB 自动转 float 故冒烟不显)
            for c in stats.columns:
                if c not in ("instrument", "d"):
                    stats[c] = pd.to_numeric(stats[c], errors="coerce").astype("float64")
            stats = stats[stats["nb"] >= min_bars]
            parts.append(finish_primitives(stats))
            del stats
            gc.collect()
            print(f"[f006-step1] 分块 {i+1}/{len(chunks)} ({cs}..{ce}): "
                  f"{len(parts[len(parts) - 1])} stock-day | 剩余内存 {_avail()}")
        if not parts:
            raise ValueError(f"build_primitives: [{d0},{d1}] 无数据")
        prim = pd.concat(parts, ignore_index=True)
        prim = prim.sort_values(["instrument", "date"]).reset_index(drop=True)
        return prim


    def panel_from_prim(prim, prim_cols, cov_min=0.5):
        dates = np.sort(prim["date"].unique())
        insts = np.sort(prim["instrument"].astype(str).unique())
        values = np.zeros((len(dates), len(insts), len(prim_cols)), dtype=np.float32)
        coverage = np.zeros((len(dates), len(insts)), dtype=np.int16)
        ordered = prim.sort_values(["date", "instrument"]).set_index(
            ["date", "instrument"], verify_integrity=True
        )
        for index, column in enumerate(prim_cols):
            pivot = ordered[column].unstack("instrument").reindex(
                index=dates, columns=insts
            )
            raw = pivot.to_numpy(np.float32)
            for row_index in range(len(dates)):
                finite = np.isfinite(raw[row_index])
                coverage[row_index, finite] += 1
                if not finite.any():
                    continue
                ranks = (
                    pd.Series(raw[row_index, finite], copy=False)
                    .rank(method="average", pct=True)
                    .to_numpy(np.float64)
                )
                mean = ranks.mean()
                std = ranks.std(ddof=0)
                values[row_index, finite, index] = (
                    (ranks - mean) / (std + 1e-8)
                ).astype(np.float32)
        valid = coverage >= int(np.ceil(len(prim_cols) * cov_min))
        return dates, insts, values, valid


    def frozen_mlp(sample, state):
        period_count = sample.shape[1]
        if period_count != 20:
            raise ValueError("frozen network expects a 20-session input")
        endpoint = sample.select(1, period_count - 1)
        five_session_block = sample.narrow(1, period_count - 5, 5)
        summary = torch.cat(
            [
                endpoint,
                five_session_block.mean(dim=1),
                sample.mean(dim=1),
                five_session_block.std(dim=1, unbiased=False),
                sample.std(dim=1, unbiased=False),
                endpoint - sample.select(1, period_count - 2),
                endpoint - sample.select(1, period_count - 6),
            ],
            dim=1,
        )
        layer1 = torch_functional.gelu(
            torch_functional.linear(summary, state["net.0.weight"], state["net.0.bias"])
        )
        layer2 = torch_functional.gelu(
            torch_functional.linear(layer1, state["net.3.weight"], state["net.3.bias"])
        )
        return torch_functional.linear(
            layer2, state["net.6.weight"], state["net.6.bias"]
        ).squeeze(1)


    def infer_factor_from_state(
        state, values, valid, window, orientation=1.0, device="cpu", min_xsec=100
    ):
        if window != 20:
            raise ValueError("frozen network expects a 20-session input")
        compute_dtype = next(iter(state.values())).dtype
        if compute_dtype not in (torch.float32, torch.float64):
            raise TypeError("network tensors must use a floating computation dtype")
        values_t = torch.as_tensor(values, dtype=compute_dtype, device=device)
        valid_t = torch.as_tensor(valid, dtype=torch.bool, device=device)
        output = np.full(valid.shape, np.nan, dtype=np.float32)
        with torch.no_grad():
            for day in range(window - 1, len(values)):
                active = valid_t.select(0, day).nonzero(as_tuple=True)[0]
                if len(active) < min_xsec:
                    continue
                block_start = day - (window - 1)
                sample = (
                    values_t.narrow(0, block_start, window)
                    .index_select(1, active)
                    .permute(1, 0, 2)
                )
                prediction = frozen_mlp(sample, state).float() * float(orientation)
                prediction = (prediction - prediction.mean()) / (
                    prediction.std(unbiased=False) + 1e-8
                )
                output[day, active.cpu().numpy()] = prediction.cpu().numpy()
        return output


    def find_weights(name):
        candidates = [
            name,
            os.path.join(os.getcwd(), name),
            os.path.join(os.path.dirname(os.getcwd()), name),
            os.path.join(os.path.expanduser("~"), "work", name),
            os.path.join("/home/aiuser/work", name),
        ]
        for path in candidates:
            if os.path.isfile(path):
                return path
        raise FileNotFoundError(f"missing weight attachment: {name}")

    with open(find_weights("f033_weights.json"), "r", encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload["features"] != PRIM_COLS:
        raise ValueError("weight feature contract does not match submission source")
    network = payload["network"]
    if network != {"input_features": 37, "summary_width": 7, "layers": [259, 192, 64, 1]}:
        raise ValueError("unexpected network contract")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    state = {
        key: torch.tensor(np.asarray(value, dtype=np.float64), device=device)
        for key, value in payload["state_dict"].items()
    }
    print(f"[f033] frozen MLP loaded | device={device} | precision=float64 | tensors={len(state)}")

    def query(sql, filters):
        return dai.query(sql, filters=filters, compression=True).df()

    start_day = str(start_date)[:10]
    end_day = str(end_date)[:10]
    warm_start = (pd.Timestamp(start_day) - pd.Timedelta(days=50)).strftime("%Y-%m-%d")
    primitives = build_primitives(
        query,
        datasources["bar1m"],
        "bigalpha_2026_instruments",
        warm_start,
        end_day,
    )
    primitives["instrument"] = primitives["instrument"].astype(str)
    gc.collect()

    dates, instruments, values, valid = panel_from_prim(primitives, PRIM_COLS)
    factor = infer_factor_from_state(
        state,
        values,
        valid,
        T,
        orientation=payload["orientation"],
        device=device,
    )
    frame = (
        pd.DataFrame(factor, index=pd.to_datetime(dates), columns=instruments)
        .rename_axis(index="date", columns="instrument")
        .stack()
        .rename("factor")
        .reset_index()
    )
    frame = frame[frame["date"].between(start_date, end_date)]

    pool = query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        {"date": [start_day, end_day]},
    )
    pool["date"] = pd.to_datetime(pool["date"])
    pool["instrument"] = pool["instrument"].astype(str)
    pool = pool[pool["date"].between(start_date, end_date)][["date", "instrument"]]
    output = pool.merge(frame, how="left", on=["date", "instrument"])
    median = output.groupby("date")["factor"].transform("median")
    output["factor"] = output["factor"].fillna(median).fillna(0.0)
    output["factor"] = output["factor"].replace([np.inf, -np.inf], 0.0)
    output = output.sort_values(["date", "instrument"]).reset_index(drop=True)
    return output[["date", "instrument", "factor"]]
